In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [3]:
import json

from prompt_toolkit import prompt


def generate_dataset():
    prompt = """
    Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
    that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
    each representing task that requires Python, JSON, or a Regex to complete.

    Example output:
    ```json
    [
        {
            "task": "Description of task"
        },
        ...additional
    ]
    ```

    * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
    * Focus on tasks that do not require writing much code

    Please generate 3 objects.
    """

    messages= []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [5]:
dataset= generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [6]:
def run_prompt(test_case):
    """Merges the prompt and the test case input, then returns the result"""
    prompt= f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages= []
    add_user_message(messages, prompt)
    output= chat(messages)
    return output

In [7]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    #TODO: - Grading
    score = 10 

    return{
        "output": output,
        "test_case": test_case,
        "score": score
    }
    

In [8]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [10]:
with open ("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [ ]:
#The evaluation returns a structured JSON array where each object represents one test case result
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Region Extraction Function\n\nHere's a comprehensive solution with multiple approaches:\n\n```python\nimport re\nfrom urllib.parse import urlparse\n\ndef extract_region_from_s3_uri(s3_uri: str) -> str:\n    \"\"\"\n    Extract AWS region from an S3 bucket URI.\n    \n    Args:\n        s3_uri: S3 URI in format 's3://bucket-name/path/to/file'\n        \n    Returns:\n        AWS region as string, or empty string if not found\n        \n    Examples:\n        >>> extract_region_from_s3_uri('s3://my-bucket-us-west-2/path/to/file')\n        'us-west-2'\n        >>> extract_region_from_s3_uri('s3://bucket-eu-central-1/file.txt')\n        'eu-central-1'\n    \"\"\"\n    # Valid AWS regions pattern\n    # Format: region-code-number (e.g., us-east-1, eu-west-2)\n    region_pattern = r'(us|eu|ap|ca|sa|af|me|cn)-(north|south|east|west|central)?-?\\d+[a-z]?'\n    \n    # Search for region pattern in the URI\n    match = re.search(region_pattern, s3_uri)\n    \n    if